# Dunnhumby K=1 M4 `q_C` 배정 대조군 개발 screen

원 LightGCN식 K=1 균등 음성 BPR에서 M1, 실제 `q_C` M4, degree-matched `q_C` 순열 M4를 같은 실행으로 새로 학습합니다.

- 학습: DAY 1~683
- 개발평가: DAY 684~690, seed 42
- 과제: 학습기간의 (user, item) 쌍을 정답과 후보에서 제외한 신규상품 추천
- 고정: binary graph, `MIN_ITEM_INTER=1`, 100 epoch, 음성 1개, 외부 재정렬 없음
- 대조: `q_C=percentile(n_u×v_u)`만 binary user-degree 10분위 안에서 섞고, 상품 구매금액 백분위와 사용자 경제구간 적합도는 고정합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'fe72f58594b518c79829677f4991ff7a65653681'
REPO_DIR = '/content/clv-m2-lightgcn-runner'
!if [ ! -d "$REPO_DIR/.git" ]; then git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git "$REPO_DIR"; fi
!git -C "$REPO_DIR" fetch -q origin $REVIEWED_SHA
!git -C "$REPO_DIR" checkout -q $REVIEWED_SHA
%cd /content/clv-m2-lightgcn-runner
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import json
import torch
import lightgcn_clv_m4_k1_assignment_control_screen as controls

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert controls.CODE_VERSION == 'm4-personalized-positive-weight-k1-assignment-control-development-screen-v1'
cfg = controls.configure_m4_k1_assignment_control_screen()
summary = controls.preflight_summary(cfg)
assert cfg.negative_count == 1
assert cfg.seed == 42
assert summary['trained_models'] == list(controls.MODEL_IDS)
assert summary['reused_models'] == []
assert summary['control']['permuted'] == 'q_C only'
assert summary['fixed']['new_item_task'] is True
assert summary['fixed']['min_item_interactions'] == 1
assert summary['fixed']['final_test_constructed'] is False
assert summary['fixed']['holdout_constructed'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
result_df = controls.run_m4_k1_assignment_control_screen(cfg)


In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

core = [
    'model_id', 'm4_assignment', 'row_weight_cv',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10', 'vndcg@10',
    'coverage@10', 'user_value_tendency_recommended_price_alignment',
]
print('1) M1·실제 M4·q_C 순열 M4 절대지표')
show(result_df[core])
print('2) M1과 순열 M4 대비 전체 지표')
show(result_df.attrs['comparison'])
print('3) q_C 순열 불변식')
print(json.dumps(result_df.attrs['control_diagnostics'], ensure_ascii=False, indent=2))
print('4) Top-10 변경 비율')
show(result_df.attrs['top10_overlap'])
print('5) 사전 고정 기준 판독')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('저장 파일:', json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
